# 04 Visualize Results

Visualize Frankfurt airspace complexity outputs from the Gold V2 layer and prepare research-facing plots for the PRU-inspired terminal workflow.


## Figure Plan

This notebook generates three core outputs for the V2 Frankfurt workflow:

1. Horizontal-cell complexity heatmap across the Frankfurt study area
2. 15-minute complexity trend for the selected run and cell scheme
3. Top hotspot ranking based on the V2 horizontal hotspot table


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import yaml

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.visualization.basemap import (
    DEFAULT_BASEMAP_NAME,
    draw_basemap,
    feature_point_coordinates,
    first_feature,
    load_basemap_geojson,
)
from src.visualization.vertical_profile import (
    build_vertical_profile_frame,
    summarize_vertical_profile,
    vertical_profile_axis_label,
)

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)

with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def parse_float(raw_value: str, *, default: float) -> float:
    text = raw_value.strip()
    return default if not text else float(text)

def parse_int(raw_value: str, *, default: int) -> int:
    text = raw_value.strip()
    return default if not text else int(text)

def parse_bool(raw_value: str, *, default: bool) -> bool:
    text = raw_value.strip().lower()
    if not text:
        return default
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    raise ValueError("Expected a boolean string.")

def parse_optional_timestamp(raw_value: str):
    text = raw_value.strip()
    if not text:
        return None
    return pd.Timestamp(text).to_pydatetime()

def format_for_id(value) -> str:
    numeric = float(value)
    if numeric.is_integer():
        return str(int(numeric))
    return str(numeric).replace('.', 'p')

def build_default_cell_scheme_id(*, focus_airport: str, horizontal_cell_nm: float, vertical_cell_ft: float, analysis_window_minutes: int, min_altitude_ft: float, max_altitude_ft: float, projection_mode: str) -> str:
    return (
        f"{focus_airport.lower()}_h{format_for_id(horizontal_cell_nm)}nm_"
        f"v{format_for_id(vertical_cell_ft)}ft_"
        f"w{analysis_window_minutes}m_"
        f"a{format_for_id(min_altitude_ft)}to{format_for_id(max_altitude_ft)}_"
        f"{projection_mode}_v2"
    )

def normalize_scalar(value):
    if value is None:
        return None
    if hasattr(value, "to_pydatetime"):
        return value.to_pydatetime()
    return value

def append_row_to_table(*, target_table: str, payload: dict) -> None:
    target_schema = spark.table(target_table).schema
    row = {field.name: normalize_scalar(payload.get(field.name)) for field in target_schema}
    spark.createDataFrame([row], schema=target_schema).write.mode("append").saveAsTable(target_table)

def latest_gold_v2_run(*, catalog: str):
    return spark.sql(
        f"""
        SELECT run_id, metadata_json
        FROM {catalog}.obs.pipeline_run_log
        WHERE pipeline_name = '03_build_complexity_metrics'
          AND layer = 'gold_v2'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()

def parse_cell_scheme_from_metadata(metadata_json: str | None):
    if not metadata_json:
        return None
    import json
    try:
        payload = json.loads(metadata_json)
    except Exception:
        return None
    return payload.get("cell_scheme_id")

def plot_square_outline(ax, row, color="#666666", linewidth=0.35):
    xs = [row["min_longitude"], row["max_longitude"], row["max_longitude"], row["min_longitude"], row["min_longitude"]]
    ys = [row["min_latitude"], row["min_latitude"], row["max_latitude"], row["max_latitude"], row["min_latitude"]]
    ax.plot(xs, ys, color=color, linewidth=linewidth, alpha=0.25)

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_source_run_id = ""
default_cell_scheme_id = ""
default_horizontal_cell_nm = "10"
default_vertical_cell_ft = "3000"
default_analysis_window_minutes = "15"
default_min_altitude_ft = "0"
default_max_altitude_ft = "24500"
default_projection_mode = "local_nm"
default_top_n = "10"
default_basemap_name = DEFAULT_BASEMAP_NAME
default_show_basemap = "true"
default_show_basemap_labels = "true"
default_show_vertical_profile = "true"
default_vertical_profile_axis = "east_west"
default_vertical_profile_window_start = ""
default_vertical_profile_distance_bin_nm = "5"

catalog = default_catalog
source_run_id_raw = default_source_run_id
cell_scheme_id_raw = default_cell_scheme_id
horizontal_cell_nm_raw = default_horizontal_cell_nm
vertical_cell_ft_raw = default_vertical_cell_ft
analysis_window_minutes_raw = default_analysis_window_minutes
min_altitude_ft_raw = default_min_altitude_ft
max_altitude_ft_raw = default_max_altitude_ft
projection_mode = default_projection_mode
top_n_raw = default_top_n
basemap_name_raw = default_basemap_name
show_basemap_raw = default_show_basemap
show_basemap_labels_raw = default_show_basemap_labels
show_vertical_profile_raw = default_show_vertical_profile
vertical_profile_axis_raw = default_vertical_profile_axis
vertical_profile_window_start_raw = default_vertical_profile_window_start
vertical_profile_distance_bin_nm_raw = default_vertical_profile_distance_bin_nm

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("source_run_id", default_source_run_id)
    dbutils.widgets.text("cell_scheme_id", default_cell_scheme_id)
    dbutils.widgets.text("horizontal_cell_nm", default_horizontal_cell_nm)
    dbutils.widgets.text("vertical_cell_ft", default_vertical_cell_ft)
    dbutils.widgets.text("analysis_window_minutes", default_analysis_window_minutes)
    dbutils.widgets.text("min_altitude_ft", default_min_altitude_ft)
    dbutils.widgets.text("max_altitude_ft", default_max_altitude_ft)
    dbutils.widgets.dropdown("projection_mode", default_projection_mode, ["local_nm"])
    dbutils.widgets.text("top_n", default_top_n)
    dbutils.widgets.text("basemap_name", default_basemap_name)
    dbutils.widgets.dropdown("show_basemap", default_show_basemap, ["true", "false"])
    dbutils.widgets.dropdown("show_basemap_labels", default_show_basemap_labels, ["true", "false"])
    dbutils.widgets.dropdown("show_vertical_profile", default_show_vertical_profile, ["true", "false"])
    dbutils.widgets.dropdown("vertical_profile_axis", default_vertical_profile_axis, ["east_west", "north_south"])
    dbutils.widgets.text("vertical_profile_window_start", default_vertical_profile_window_start)
    dbutils.widgets.text("vertical_profile_distance_bin_nm", default_vertical_profile_distance_bin_nm)

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    source_run_id_raw = dbutils.widgets.get("source_run_id").strip()
    cell_scheme_id_raw = dbutils.widgets.get("cell_scheme_id").strip()
    horizontal_cell_nm_raw = dbutils.widgets.get("horizontal_cell_nm").strip() or default_horizontal_cell_nm
    vertical_cell_ft_raw = dbutils.widgets.get("vertical_cell_ft").strip() or default_vertical_cell_ft
    analysis_window_minutes_raw = dbutils.widgets.get("analysis_window_minutes").strip() or default_analysis_window_minutes
    min_altitude_ft_raw = dbutils.widgets.get("min_altitude_ft").strip() or default_min_altitude_ft
    max_altitude_ft_raw = dbutils.widgets.get("max_altitude_ft").strip() or default_max_altitude_ft
    projection_mode = dbutils.widgets.get("projection_mode").strip() or default_projection_mode
    top_n_raw = dbutils.widgets.get("top_n").strip() or default_top_n
    basemap_name_raw = dbutils.widgets.get("basemap_name").strip() or default_basemap_name
    show_basemap_raw = dbutils.widgets.get("show_basemap")
    show_basemap_labels_raw = dbutils.widgets.get("show_basemap_labels")
    show_vertical_profile_raw = dbutils.widgets.get("show_vertical_profile")
    vertical_profile_axis_raw = dbutils.widgets.get("vertical_profile_axis").strip() or default_vertical_profile_axis
    vertical_profile_window_start_raw = dbutils.widgets.get("vertical_profile_window_start").strip()
    vertical_profile_distance_bin_nm_raw = dbutils.widgets.get("vertical_profile_distance_bin_nm").strip() or default_vertical_profile_distance_bin_nm

focus_airport = region_config["focus_airport"]
bbox = region_config["regional_bbox"]
horizontal_cell_nm = parse_float(horizontal_cell_nm_raw, default=10.0)
vertical_cell_ft = parse_float(vertical_cell_ft_raw, default=3000.0)
analysis_window_minutes = parse_int(analysis_window_minutes_raw, default=15)
min_altitude_ft = parse_float(min_altitude_ft_raw, default=0.0)
max_altitude_ft = parse_float(max_altitude_ft_raw, default=24500.0)
top_n = parse_int(top_n_raw, default=10)
selected_basemap_name = basemap_name_raw.strip() or DEFAULT_BASEMAP_NAME
show_basemap = parse_bool(show_basemap_raw, default=True)
show_basemap_labels = parse_bool(show_basemap_labels_raw, default=True)
show_vertical_profile = parse_bool(show_vertical_profile_raw, default=True)
vertical_profile_axis = vertical_profile_axis_raw.strip().lower()
selected_vertical_profile_window_start = parse_optional_timestamp(vertical_profile_window_start_raw)
vertical_profile_distance_bin_nm = parse_float(vertical_profile_distance_bin_nm_raw, default=5.0)
if top_n <= 0:
    raise ValueError("top_n must be positive")
if vertical_profile_axis not in {"east_west", "north_south"}:
    raise ValueError("vertical_profile_axis must be east_west or north_south")
if vertical_profile_distance_bin_nm <= 0:
    raise ValueError("vertical_profile_distance_bin_nm must be positive")

default_cell_scheme_id = build_default_cell_scheme_id(
    focus_airport=focus_airport,
    horizontal_cell_nm=horizontal_cell_nm,
    vertical_cell_ft=vertical_cell_ft,
    analysis_window_minutes=analysis_window_minutes,
    min_altitude_ft=min_altitude_ft,
    max_altitude_ft=max_altitude_ft,
    projection_mode=projection_mode,
)

horizontal_complexity_table_v2 = f"{catalog}.gld_airspace.horizontal_complexity_v2"
hotspots_table_v2 = f"{catalog}.gld_airspace.horizontal_hotspots_v2"
trend_table_v2 = f"{catalog}.gld_airspace.complexity_trend_v2"
silver_table_v2 = f"{catalog}.slv_airspace.flight_states_cellized_v2"
airspace_cells_table_v2 = f"{catalog}.ref.airspace_cells_v2"
pipeline_log_table = f"{catalog}.obs.pipeline_run_log"
source_run_id_input = source_run_id_raw.strip()
cell_scheme_id_input = cell_scheme_id_raw.strip()

basemap_geojson = load_basemap_geojson(selected_basemap_name)
basemap_feature_count = len(basemap_geojson["features"])
airport_point_feature = first_feature(basemap_geojson, layer_kind="airport_point")
if airport_point_feature is None:
    raise ValueError("Selected basemap does not contain an airport_point feature.")
airport_longitude, airport_latitude = feature_point_coordinates(airport_point_feature)

plt.style.use("ggplot")
pd.set_option("display.max_columns", 30)


In [ ]:
latest_gold_run = latest_gold_v2_run(catalog=catalog)

if source_run_id_input:
    selected_source_run_id = source_run_id_input
else:
    if latest_gold_run is None:
        raise ValueError("No successful V2 Gold run found. Run 03_build_complexity_metrics.ipynb first.")
    selected_source_run_id = latest_gold_run["run_id"]

selected_cell_scheme_id = cell_scheme_id_input
if not selected_cell_scheme_id:
    if source_run_id_input:
        selected_cell_scheme_id = default_cell_scheme_id
    else:
        selected_cell_scheme_id = parse_cell_scheme_from_metadata(latest_gold_run["metadata_json"]) or default_cell_scheme_id

horizontal_complexity_df_v2 = (
    spark.table(horizontal_complexity_table_v2)
    .where(F.col("run_id") == F.lit(selected_source_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
hotspots_df_v2 = (
    spark.table(hotspots_table_v2)
    .where(F.col("run_id") == F.lit(selected_source_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
trend_df_v2 = (
    spark.table(trend_table_v2)
    .where(F.col("run_id") == F.lit(selected_source_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
silver_df_v2 = (
    spark.table(silver_table_v2)
    .where(F.col("run_id") == F.lit(selected_source_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
airspace_cells_df_v2 = (
    spark.table(airspace_cells_table_v2)
    .where(F.col("run_id") == F.lit(selected_source_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)

horizontal_cell_ref_df = (
    airspace_cells_df_v2
    .groupBy("horizontal_cell_id")
    .agg(
        F.min("min_latitude").alias("min_latitude"),
        F.max("max_latitude").alias("max_latitude"),
        F.min("min_longitude").alias("min_longitude"),
        F.max("max_longitude").alias("max_longitude"),
        F.avg("center_latitude").alias("center_latitude"),
        F.avg("center_longitude").alias("center_longitude"),
        F.max("horizontal_cell_nm").alias("horizontal_cell_nm"),
        F.countDistinct("vertical_cell_id").alias("vertical_layers_defined")
    )
)

horizontal_rows = horizontal_complexity_df_v2.count()
hotspot_rows = hotspots_df_v2.count()
trend_rows = trend_df_v2.count()
silver_rows = silver_df_v2.count()
cell_ref_rows = horizontal_cell_ref_df.count()

if horizontal_rows == 0 or hotspot_rows == 0 or trend_rows == 0 or silver_rows == 0:
    raise ValueError(
        f"V2 Gold rows missing for run_id={selected_source_run_id} and cell_scheme_id={selected_cell_scheme_id}. Run 03_build_complexity_metrics.ipynb first."
    )

if selected_vertical_profile_window_start is None:
    peak_window_row = trend_df_v2.orderBy(F.col("peak_complexity_score").desc(), F.col("window_start").asc()).first()
    if peak_window_row is None:
        raise ValueError("No trend window is available for vertical profile plotting.")
    selected_vertical_profile_window_start = peak_window_row["window_start"]

summary_row = horizontal_complexity_df_v2.agg(
    F.countDistinct("horizontal_cell_id").alias("horizontal_cell_count"),
    F.countDistinct("window_start").alias("window_count"),
    F.min("window_start").alias("min_window_start"),
    F.max("window_start").alias("max_window_start"),
    F.min("complexity_score").alias("min_complexity_score"),
    F.max("complexity_score").alias("max_complexity_score")
).first()

run_summary = {
    "catalog": catalog,
    "focus_airport": focus_airport,
    "source_run_id": selected_source_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "top_n": top_n,
    "horizontal_rows": horizontal_rows,
    "hotspot_rows": hotspot_rows,
    "trend_rows": trend_rows,
    "silver_rows": silver_rows,
    "cell_ref_rows": cell_ref_rows,
    "horizontal_cell_count": int(summary_row["horizontal_cell_count"]),
    "window_count": int(summary_row["window_count"]),
    "min_window_start": str(summary_row["min_window_start"]),
    "max_window_start": str(summary_row["max_window_start"]),
    "min_complexity_score": float(summary_row["min_complexity_score"]),
    "max_complexity_score": float(summary_row["max_complexity_score"]),
    "show_basemap": show_basemap,
    "show_basemap_labels": show_basemap_labels,
    "basemap_name": selected_basemap_name if show_basemap else None,
    "basemap_feature_count": basemap_feature_count,
    "show_vertical_profile": show_vertical_profile,
    "vertical_profile_axis": vertical_profile_axis,
    "vertical_profile_window_start": str(selected_vertical_profile_window_start),
    "vertical_profile_distance_bin_nm": vertical_profile_distance_bin_nm,
    "bbox": bbox,
}

run_summary


In [ ]:
heatmap_spark_df = (
    horizontal_complexity_df_v2
    .groupBy("horizontal_cell_id")
    .agg(
        F.avg("complexity_score").alias("avg_complexity_score"),
        F.max("complexity_score").alias("peak_complexity_score"),
        F.avg("aircraft_count").alias("avg_aircraft_count"),
        F.avg("active_vertical_cells").alias("avg_active_vertical_cells")
    )
    .join(horizontal_cell_ref_df, on="horizontal_cell_id", how="inner")
    .orderBy(F.col("avg_complexity_score").desc())
)

hotspots_spark_df = (
    hotspots_df_v2
    .join(horizontal_cell_ref_df, on="horizontal_cell_id", how="left")
    .orderBy("ranking")
)

trend_spark_df = trend_df_v2.orderBy("window_start")
preview_spark_df = (
    horizontal_complexity_df_v2
    .orderBy(F.col("complexity_score").desc(), F.col("window_start").asc())
    .limit(20)
)
vertical_profile_source_spark_df = (
    silver_df_v2
    .where(F.col("analysis_window_start") == F.lit(selected_vertical_profile_window_start))
    .join(
        airspace_cells_df_v2.select("cell_id", "min_altitude_ft", "max_altitude_ft").dropDuplicates(["cell_id"]),
        on="cell_id",
        how="left",
    )
    .select(
        "analysis_window_start",
        "event_time",
        "icao24",
        "cell_id",
        "longitude",
        "latitude",
        "altitude_ft",
        "min_altitude_ft",
        "max_altitude_ft",
    )
)

heatmap_df = heatmap_spark_df.toPandas()
trend_plot_df = trend_spark_df.toPandas()
hotspots_plot_df = hotspots_spark_df.limit(top_n).toPandas()
vertical_profile_source_df = vertical_profile_source_spark_df.toPandas()
vertical_profile_df = build_vertical_profile_frame(
    vertical_profile_source_df,
    origin_longitude=airport_longitude,
    origin_latitude=airport_latitude,
    axis=vertical_profile_axis,
    distance_bin_nm=vertical_profile_distance_bin_nm,
)
vertical_profile_summary = summarize_vertical_profile(vertical_profile_df)
preview_payload = {
    "heatmap_rows": int(len(heatmap_df)),
    "trend_rows": int(len(trend_plot_df)),
    "hotspot_rows": int(len(hotspots_plot_df)),
    "vertical_profile_rows": int(vertical_profile_summary["profile_rows"]),
    "vertical_profile_distance_bins": int(vertical_profile_summary["distance_bins"]),
    "vertical_profile_altitude_bands": int(vertical_profile_summary["altitude_bands"]),
}

if heatmap_df.empty or trend_plot_df.empty or hotspots_plot_df.empty:
    raise ValueError("Visualization inputs are empty after converting to pandas.")
if show_vertical_profile and vertical_profile_df.empty:
    raise ValueError("Vertical profile inputs are empty for the selected analysis window.")

trend_plot_df["window_start"] = pd.to_datetime(trend_plot_df["window_start"])
hotspots_plot_df = hotspots_plot_df.sort_values("ranking", ascending=False)

preview_payload


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
ax.set_facecolor("#f7f4ec")
if show_basemap and basemap_geojson is not None:
    draw_basemap(
        ax,
        basemap_geojson,
        layer_kinds={"simplified_terminal_area", "airport_reference_zone"},
        annotate=False,
    )

for _, row in heatmap_df.iterrows():
    plot_square_outline(ax, row)

scatter = ax.scatter(
    heatmap_df["center_longitude"],
    heatmap_df["center_latitude"],
    c=heatmap_df["avg_complexity_score"],
    s=260,
    marker="s",
    cmap="YlOrRd",
    alpha=0.88,
    edgecolors="none",
)
if show_basemap and basemap_geojson is not None:
    draw_basemap(
        ax,
        basemap_geojson,
        layer_kinds={"runway_centerline", "airport_point"},
        annotate=show_basemap_labels,
    )
ax.set_title(
    f"Frankfurt Horizontal Complexity Heatmap (V2)\n"
    f"run_id={selected_source_run_id} | scheme={selected_cell_scheme_id}"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(float(bbox["min_longitude"]), float(bbox["max_longitude"]))
ax.set_ylim(float(bbox["min_latitude"]), float(bbox["max_latitude"]))
cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Average Complexity Score")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trend_plot_df["window_start"], trend_plot_df["avg_complexity_score"], marker="o", linewidth=2, label="Average")
ax.plot(trend_plot_df["window_start"], trend_plot_df["peak_complexity_score"], marker="s", linewidth=2, label="Peak")
ax.set_title(
    f"Frankfurt 15-minute Complexity Trend (V2)\n"
    f"run_id={selected_source_run_id} | scheme={selected_cell_scheme_id}"
)
ax.set_xlabel("Window Start (UTC)")
ax.set_ylabel("Complexity Score")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(hotspots_plot_df["horizontal_cell_id"], hotspots_plot_df["avg_complexity_score"], color="#d95f02")
ax.set_title(
    f"Top {top_n} Frankfurt Horizontal Hotspots (V2)\n"
    f"run_id={selected_source_run_id} | scheme={selected_cell_scheme_id}"
)
ax.set_xlabel("Average Complexity Score")
ax.set_ylabel("Horizontal Cell ID")
plt.tight_layout()
plt.show()


In [ ]:
if show_vertical_profile:
    fig, ax = plt.subplots(figsize=(11, 6))
    scatter = ax.scatter(
        vertical_profile_df["distance_bin_center_nm"],
        vertical_profile_df["altitude_band_mid_ft"],
        c=vertical_profile_df["aircraft_count"],
        s=90 + (vertical_profile_df["active_cell_count"] * 60),
        marker="s",
        cmap="YlOrRd",
        alpha=0.9,
        edgecolors="white",
        linewidths=0.35,
    )
    ax.axvline(0.0, color="#3a3a3a", linestyle="--", linewidth=1.0, alpha=0.7)
    ax.set_title(
        f"Frankfurt Vertical Profile (V2)\n"
        f"window_start={selected_vertical_profile_window_start} | axis={vertical_profile_axis}"
    )
    ax.set_xlabel(vertical_profile_axis_label(vertical_profile_axis))
    ax.set_ylabel("Altitude Band Midpoint (ft)")
    ax.text(0.01, 0.97, "EDDF", transform=ax.transAxes, ha="left", va="top", fontsize=10, color="#333333")
    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label("Aircraft Count In Distance-Altitude Bin")
    plt.tight_layout()
    plt.show()


In [ ]:
display(heatmap_spark_df.limit(20))
display(trend_spark_df)
display(hotspots_spark_df.limit(top_n))
if show_vertical_profile:
    display(vertical_profile_source_spark_df.limit(50))
display(preview_spark_df)


In [ ]:
final_summary = {
    "status": "success",
    "source_run_id": selected_source_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "horizontal_rows": horizontal_rows,
    "hotspot_rows": hotspot_rows,
    "trend_rows": trend_rows,
    "horizontal_cell_count": run_summary["horizontal_cell_count"],
    "window_count": run_summary["window_count"],
    "top_n": top_n,
    "heatmap_points": preview_payload["heatmap_rows"],
    "show_basemap": show_basemap,
    "basemap_name": selected_basemap_name if show_basemap else None,
    "show_vertical_profile": show_vertical_profile,
    "vertical_profile_window_start": str(selected_vertical_profile_window_start),
    "vertical_profile_rows": preview_payload["vertical_profile_rows"],
}

final_summary
